# DeepFake Detection Agent - Cloud GPU Training Pipeline
This notebook pulls qualified Hugging Face datasets from `training_queue.json`, fine-tunes the Vision Transformer (ViT) DeepFake Classifier on Google Colab GPUs (T4 / A100), performs temperature scaling calibration, and exports detailed evaluation reports.

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets torch torchvision accelerate huggingface_hub scikit-learn onnx onnxruntime

In [ ]:
# 2. Clone Repository & Robustly Load Training Queue
import os, sys, json, getpass

# Ensure working directory is correctly positioned
if not os.path.exists('dataset_pipeline/training_queue.json'):
    try:
        from google.colab import userdata
        gh_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        gh_token = None

    if not gh_token:
        try:
            gh_token = getpass.getpass('Enter your GitHub Personal Access Token (PAT) [Press Enter if public]: ')
        except Exception:
            gh_token = ''

    if gh_token.strip():
        repo_url = f'https://{gh_token.strip()}@github.com/ThangCao106/DeepFake-Detection-Agent.git'
    else:
        repo_url = 'https://github.com/ThangCao106/DeepFake-Detection-Agent.git'

    !git clone {repo_url} || git clone https://github.com/ThangCao106/NCS.git

    if os.path.exists('DeepFake-Detection-Agent'):
        os.chdir('DeepFake-Detection-Agent')
    elif os.path.exists('NCS/DeepFake-Detection-Agent'):
        os.chdir('NCS/DeepFake-Detection-Agent')

# Auto-generate training_queue.json if not present
queue_file = 'dataset_pipeline/training_queue.json'
if not os.path.exists(queue_file):
    print(f'[info] {queue_file} not found. Running HF Dataset Scout to build queue...')
    !python hf_dataset_scout.py --search deepfake --max-results 20

if os.path.exists(queue_file):
    with open(queue_file, 'r', encoding='utf-8') as f:
        queue = json.load(f)
    print(f'Successfully loaded {len(queue)} qualified datasets from training queue:')
    for item in queue:
        print(f"  - {item['dataset_id']} (License: {item['license_type']}, Samples: {item.get('sample_count', 0)})")
else:
    print('[warning] Could not load training queue. Using default fallback datasets.')
    queue = [{'dataset_id': 'Sowaiba01/Deepfake', 'license_type': 'PERMISSIVE'}]

In [ ]:
# 3. Execute 8-Stage ViT Fine-Tuning, Calibration & ONNX Export Pipeline
import torch, time, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device for training: {device}')

target_ds_id = queue[0]['dataset_id'] if len(queue) > 0 else 'Sowaiba01/Deepfake'
print(f'Target dataset selected for fine-tuning: {target_ds_id}')

start_time = time.time()

# Execute Data Pipeline, ViT Fine-Tuning, Calibration, ONNX Export, and OOD Benchmark
!python data_pipeline.py
!python train_pipeline.py
!python calibrate.py
!python export_onnx.py
!python eval_benchmark.py

training_time = round(time.time() - start_time, 2)
print(f'Training & Pipeline Execution completed in {training_time} seconds!')

In [ ]:
# 4. Generate & Display Detailed Training & Evaluation Audit Report
import json, os
from datetime import datetime

bench_file = 'custom_vit_model/benchmark_results.json'
calib_file = 'custom_vit_model/calibration_params.json'

bench_data = {}
if os.path.exists(bench_file):
    with open(bench_file, 'r') as f:
        bench_data = json.load(f)

calib_data = {}
if os.path.exists(calib_file):
    with open(calib_file, 'r') as f:
        calib_data = json.load(f)

report = {
    'timestamp': datetime.now().isoformat(),
    'compute_device': device,
    'dataset_trained': target_ds_id,
    'base_model': 'dima806/deepfake_vs_real_image_detection',
    'total_test_samples': bench_data.get('total', 0),
    'correct_predictions': bench_data.get('correct', 0),
    'out_of_distribution_accuracy': f"{bench_data.get('accuracy_percent', 0.0)}%",
    'temperature_scaling_T': calib_data.get('temperature_scaling_T', 1.0),
    'optimal_threshold_tau': calib_data.get('optimal_threshold_tau', 0.5),
    'onnx_export_status': 'SUCCESS' if os.path.exists('custom_vit_model/custom_deepfake_vit.onnx') else 'NOT_EXPORTED'
}

report_path = 'colab_training_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print('\n' + '='*60)
print('📊 DEEPFAKE DETECTION MODEL TRAINING & AUDIT REPORT')
print('='*60)
print(f"📅 Execution Date: {report['timestamp']}")
print(f"🖥️ Compute Hardware: {report['compute_device'].upper()}")
print(f"🎯 Target Dataset: {report['dataset_trained']}")
print(f"🤖 Base Model: {report['base_model']}")
print(f"📈 OOD Benchmark Test Accuracy: {report['out_of_distribution_accuracy']}")
print(f"🌡️ Temperature Scaling T*: {report['temperature_scaling_T']:.4f}")
print(f"⚖️ Optimal Decision Threshold tau*: {report['optimal_threshold_tau']:.4f}")
print(f"📦 ONNX Export Status: {report['onnx_export_status']}")
print('='*60)
print(f'Saved audit report to {report_path}')

In [ ]:
# 5. Push Trained Model Checkpoints & Report to Hugging Face Hub / Cloud
import os, getpass

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None

if not hf_token:
    try:
        hf_token = getpass.getpass('Enter your Hugging Face Write Token (HF_TOKEN) [Press Enter to skip upload]: ')
    except Exception:
        hf_token = ''

if hf_token.strip():
    os.environ['HF_TOKEN'] = hf_token.strip()
    !python upload_model_cloud.py --repo-id ThangCao106/custom-deepfake-vit
else:
    print('Skipped cloud upload. Fine-tuned model checkpoint saved locally in custom_vit_model/')